In [1]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv('../data/processed/telco_features.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,No_internet_service,No_phone_service
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,1,0,0,1
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,1,0,0,0,1,0,0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,1,0,0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,1,0,0,0,0,0,1
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,1,0,0,0


## Class Imbalance


~27% churners. With imbalance, **accuracy is misleading** (predicting "no churn" for everyone is ~73% accurate and useless). Class weighting + threshold tuning is enough here — no aggressive oversampling needed.

In [3]:
df['Churn'].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

### Why recall matters most in churn

The costs are **asymmetric**:
- **False Negative** — predict "won't churn" but they leave → you miss the chance to retain them (expensive: lost customer).
- **False Positive** — predict "will churn" but they stay → you spend a retention offer unnecessarily (cheap).

Missing a churner usually costs more than a wasted offer → prioritize **recall**.


Typical priority:
- Retention campaigns **cheap** → maximize recall (catch every possible churner).
- Retention campaigns **expensive** → balance precision/recall via F1.


In [4]:
# Train/test split (stratified to preserve class ratio)
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

THRESHOLD = 0.3  # lower than 0.5 to boost recall (tuned per model below)

## RandomForest

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.892     0.755     0.818      1033
           1      0.525     0.749     0.617       374

    accuracy                          0.753      1407
   macro avg      0.709     0.752     0.718      1407
weighted avg      0.795     0.753     0.765      1407



In [20]:
proba = rf.predict_proba(X_test)[:, 1]

print('Threshold Tuning for RandomForest for class 1:''\n')
print(f"{'thresh':<8}{'precision':<15}{'recall':<8}{'f1-score':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<15.3f}{rec:<8.3f}{f1:<8.3f}")

Threshold Tuning for RandomForest for class 1:

thresh  precision      recall  f1-score
0.25    0.481          0.783   0.596   
0.3     0.525          0.749   0.617   
0.35    0.543          0.687   0.607   
0.4     0.578          0.623   0.600   
0.45    0.619          0.561   0.589   
0.5     0.622          0.497   0.553   


## LightGBM

In [13]:
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

start_train = time.time()
lgbm.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"training time: {train_time:.2f} seconds")

start_pred = time.time()
proba = lgbm.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)
pred_time = time.time() - start_pred
print(f"prediction time: {pred_time:.4f} secs")

print(classification_report(y_test, y_pred, digits=3))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 627
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
training time: 2.03 seconds
prediction time: 0.0055 secs
              precision    recall  f1-score   support

           0      0.913     0.680     0.779      1033
           1      0.481     0.821     0.607       374

    accuracy                          0.717      1407
   macro avg      0.697     0.750     0.693      1407
weighted avg      0.798     0.717     0

In [21]:
proba = lgbm.predict_proba(X_test)[:, 1]

print('Threshold Tuning for LightGBM for class 1:''\n')
print(f"{'thresh':<8}{'precision':<15}{'recall':<8}{'f1-score':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<15.3f}{rec:<8.3f}{f1:<8.3f}")

Threshold Tuning for LightGBM for class 1:

thresh  precision      recall  f1-score
0.25    0.470          0.842   0.603   
0.3     0.481          0.821   0.607   
0.35    0.501          0.797   0.615   
0.4     0.503          0.775   0.611   
0.45    0.516          0.749   0.611   
0.5     0.526          0.703   0.602   


## XGBoost  

In [24]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)

start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"training time: {train_time:.2f} secs")

start_pred = time.time()
proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)
pred_time = time.time() - start_pred
print(f"prediction time: {pred_time:.4f} secs")

print(classification_report(y_test, y_pred, digits=3))

training time: 0.63 secs
prediction time: 0.0023 secs
              precision    recall  f1-score   support

           0      0.906     0.673     0.772      1033
           1      0.472     0.807     0.596       374

    accuracy                          0.709      1407
   macro avg      0.689     0.740     0.684      1407
weighted avg      0.791     0.709     0.725      1407



In [25]:
proba = xgb.predict_proba(X_test)[:, 1]

print('Threshold Tuning for LightGBM for class 1:''\n')
print(f"{'thresh':<8}{'precision':<15}{'recall':<8}{'f1-score':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<15.3f}{rec:<8.3f}{f1:<8.3f}")

Threshold Tuning for LightGBM for class 1:

thresh  precision      recall  f1-score
0.25    0.454          0.829   0.587   
0.3     0.472          0.807   0.596   
0.35    0.488          0.786   0.602   
0.4     0.496          0.746   0.596   
0.45    0.507          0.703   0.589   
0.5     0.528          0.679   0.594   


## Model choice: XGBoost
- Recall comparable to LightGBM (both beat RandomForest).
- **~3x faster to train** than LightGBM here.